<a href="https://colab.research.google.com/github/Jose-Codes/ML-Papers/blob/main/Gemma_1/Gemma_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 1

In [1]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.linen.initializers import variance_scaling
import math
from typing import Optional, Tuple

Model Paramters:

Gemma 1 2B:

* d_model = 2048
* Layers = 18
* Feedforward hidden dims = 32768
* Num heads = 8
* Num KV heads = 1
* Head size = 256
* Vocab size = 256128


Implementation plan:

We need to build the most fundamental pieces first and then put them all together,

First create:

* RMSNorm
* GeGLU activation function
* RoPE embeddings
* Multi Query Attention

Second,

* Transformer Decoder block

Third and lastly,

* String the model together using many Transformer blocks.







## Gemma Configuration

In [2]:
class GemmaConfig:
  """Holds all parameters for the Gemma Model."""
  def __init__(self, **kwargs):
    self.vocab_size = kwargs.get("vocab_size", 256128)
    self.d_model = kwargs.get("d_model", 2048)
    self.num_layers = kwargs.get("num_layers", 18)
    self.num_heads = kwargs.get("num_heads", 8)
    self.num_kv_heads = kwargs.get("num_kv_heads", 1)
    self.head_dim = kwargs.get("head_dim", self.d_model / self.num_heads)
    self.ffn_hidden_dim = kwargs.get('ffn_hidden_dim', 32768)
    self.rems_norm_eps = kwargs.get("rms_norm_eps", 1e-6)
    # Ensure head_dim is correctly calculated if not provided
    assert self.d_model % self.num_heads == 0, "d_model must be divisible by num_heads"

In [3]:
class RMSNorm(nn.Module):
  """
  Root Mean Squared Normalization. A simpler and more efficient alternative to LayerNorm.
  """
  dim: int
  eps: float = 1e-6

  @nn.compact
  def __call__(self, x):
    # Using float 32 for high precision
    x = x.astype(jnp.float21)
    # Calculate the reciprocal of the root mean square
    rsqrt = jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, # axis=-1 is the last dimension which is taken to be the feature dimension of the vector.
                                  keepdims=True) + self.eps) # 1 / sqrt(x^2 + e)
    x_normalized = x * rsqrt

    # Learnable scaling parameter, often called 'gamma' or 'weight'
    # Gemma multiplies by (1 + weight)
    scale = self.param('scale', nn.initializers.zeros, (self.dim,))

    return x_normalized * (1 + scale)

In [5]:
class GeGLU(nn.Module):
  """
  Gated Linear Unit with GELU activation, used in the FFN
  """
  config: GemmaConfig

  @nn.compact
  def __call__(self, x):
    # Project to the hidden dimension for both the gate and the main path
    gate_proj = nn.Dense(self.config.ffn_hidden_dim, use_bias=False,
                      kernel_init=variance_scaling(1.0, "fan_in", "normal"))(x)
    up_proj = nn.Dense(self.config.ffn_hidden_dim, use_bias=False,
                      kernel_init=variance_scaling(1.0, "fan_in", "normal"))(x)

    # The core of GeGLU: element-wise product of GELU-activated gate and
    # up-projection
    gated_output = nn.gelu(gate_proj) * up_proj

    # Project back down to the model dimension
    down_proj = nn.Dense(self.config.d_model, use_bias=False,
            kernel_init=variance_scaling(1.0, "fan_in", "normal"))(gated_output)

    return down_proj



In [ ]:
class ROPE:
  """
  Rotary Positional Embeddings (RoPE) to encode word positions.
  """

  dim: int
  max_seq_length: int = 4096

  def setup(self):
    # Inverse frequencies for the sinusoidal calculation
    inv_freq = 1.0 / (10000 ** (jnp.arrange(0, self.dim, 2).astype(jnp.float32) / self.dim))
    t = jnp.arange(self.max_seq_length)
    freqs = jnp.einsum('i,j->ij', t, inv_freq)

    # Store freqs as a buffer that is not a trainable parameter
    self.freqs = jnp.concatenate((freqs, freqs), axis=-1)

  def __call__(self, x, seq_len: int):
    # x shape: (batch, seq_len, num_heads, head_dim)

In [ ]:
class MQA:
  # TODO fill in class
  def __init__(self):
    """hello world"""

In [ ]:
class TransformerDecoderBlock:
  # TODO fill in class
  def __init__(self):
    """hello world"""

In [ ]:
class Gemma1Model:
  # TODO fill in class
  def __init__(self):
    """hello world"""